# Lab 6 — SLAM Script Deployment, and Submission files

## Part 1: Environment Verification and Workspace Setup (ROS1)

#### Connect to the Robot
1. Connect your computer to the robot's Wi-Fi network: `HW_2DF5BEE9` (Default password: `hiwonder`).
2. Open your computer terminal and connect via SSH:
   ```bash
   ssh jetauto@192.168.149.1
   # Use password: hiwonder
   ```
3. Check if ROS is working and see the active topics:
   ```bash
   rostopic list
   rosnode list
   ```
4. Start the robot's hardware base controller:
   ```bash
   roslaunch jetauto_controller jetauto_controller.launch
   ```

```bash
# 1. Verify the workspace directory paths
ls ~/jetauto_ws
ls ~/jetauto_ws/src

# 2. Source the ROS environment variables
source /opt/ros/melodic/setup.bash
source ~/jetauto_ws/devel/setup.bash

# 3. Check if the required ROS packages can be found
rospack find jetauto_slam
rospack find jetauto_gazebo
rospack find map_server
```

## Safety-First Checklist

Before running SLAM on the real robot, confirm the following:

- The robot is on the floor, not on a table or chair.
- The mapping area is clear of loose cables, bags, chairs, and people walking close to the robot.
- The battery is sufficiently charged.
- The emergency stop or power switch is accessible.
- The operator can stop the robot quickly.
- The robot will move slowly during mapping.
- The LiDAR is unobstructed and spinning/active.
- One student is assigned to watch the robot while another operates the terminal or keyboard controller.

> **Note:** SLAM quality usually improves when the robot moves slowly and smoothly. Fast turns, sharp rotations, and repeated collisions often produce distorted maps.

## Part 2: Mapping Workflows (Simulation or Physical Robot)

### Option A: Gazebo Simulation Mapping Steps
1. **Terminal 1 (Launch Environment)**: `roslaunch jetauto_gazebo room_worlds.launch`
2. **Terminal 2 (Start SLAM)**: `roslaunch jetauto_slam slam.launch sim:=true`
3. **Terminal 3 (Open RViz)**: `roslaunch jetauto_slam rviz_slam.launch sim:=true`

### Option B: Physical Robot Mapping Steps
1. **Terminal 1 (SSH Connection)**: `ssh jetauto@<robot_ip_address>`
2. **Terminal 2 (Stop Default App Service)**: `sudo systemctl stop start_app_node.service`
3. **Terminal 3 (Launch SLAM on Robot)**: `roslaunch jetauto_slam slam.launch slam_methods:=gmapping`
4. **Terminal 4 (Open RViz via Remote Desktop or Laptop)**: `roslaunch jetauto_slam rviz_slam.launch slam_methods:=gmapping`

## Inspect ROS Topics During SLAM

Use these commands to confirm that SLAM is receiving sensor and motion data. Run them while SLAM is active (simulation or physical robot).

### List all active topics
```bash
rostopic list
```

### Check key topics (one-shot echo)
```bash
rostopic echo /scan -n 1
rostopic echo /map -n 1
rostopic echo /odom -n 1
```

### View the TF tree
```bash
rosrun tf view_frames
```

### Check TF relationships
```bash
rosrun tf tf_echo map base_link
rosrun tf tf_echo odom base_link
```

### Namespaced robots
If the robot is namespaced, the topics may look like:

```
/jetauto_1/scan
/jetauto_1/map
/jetauto_1/odom
/jetauto_1/cmd_vel
```

To find the exact topic names:
```bash
rostopic list | grep scan
rostopic list | grep map
rostopic list | grep odom
rostopic list | grep cmd_vel
```

### Verification checkpoint
Record the actual topic names used in your environment:
- Scan topic: `/scan`
- Map topic: `/map`
- Odometry topic: `/odom`
- Command velocity topic: `/cmd_vel`

## Part 3: Automatically Generate the ROS1 Slow Mapping Controller

Driving the robot too fast or turning too sharply can distort your SLAM map. Run the Python cell below to automatically create and write the `slow_mapping_controller.py` script into your workspace directory.

In [ ]:
import os

# Define the destination paths
pkg_dir = os.path.expanduser('~/jetauto_ws/src/lab6_slam_control')
script_dir = os.path.join(pkg_dir, 'scripts')

# Create the ROS1 package folder structure
os.makedirs(script_dir, exist_ok=True)

ros1_script_code = """#!/usr/bin/env python
import sys
import termios
import tty
import rospy
from geometry_msgs.msg import Twist

HELP = \"\"\"
Slow SLAM mapping controller (ROS1)
----------------------------
w: forward slowly
s: backward slowly
a: rotate left slowly
d: rotate right slowly
x: stop
q: quit
\"\"\"

def get_key():
    settings = termios.tcgetattr(sys.stdin)
    try:
        tty.setraw(sys.stdin.fileno())
        key = sys.stdin.read(1)
    finally:
        termios.tcsetattr(sys.stdin, termios.TCSADRAIN, settings)
    return key

def main():
    rospy.init_node('slow_mapping_controller')
    cmd_topic = rospy.get_param('~cmd_topic', '/cmd_vel')
    pub = rospy.Publisher(cmd_topic, Twist, queue_size=10)

    linear_speed = rospy.get_param('~linear_speed', 0.08)
    angular_speed = rospy.get_param('~angular_speed', 0.25)

    print(HELP)
    rate = rospy.Rate(10)

    while not rospy.is_shutdown():
        key = get_key()
        msg = Twist()

        if key == 'w':
            msg.linear.x = linear_speed
        elif key == 's':
            msg.linear.x = -linear_speed
        elif key == 'a':
            msg.angular.z = angular_speed
        elif key == 'd':
            msg.angular.z = -angular_speed
        elif key == 'x':
            pass
        elif key == 'q':
            break
        else:
            continue

        pub.publish(msg)
        rate.sleep()

    pub.publish(Twist())

if __name__ == '__main__':
    main()
"""

script_file_path = os.path.join(script_dir, 'slow_mapping_controller.py')
with open(script_file_path, 'w') as f:
    f.write(ros1_script_code.strip())

print(f"[Success] ROS1 slow keyboard control script written to: {script_file_path}")

### Compile and Run the Keyboard Controller (ROS1)

Run these commands in your system terminal to make the script executable, compile the package, and launch the node:
```bash
chmod +x ~/jetauto_ws/src/lab6_slam_control/scripts/slow_mapping_controller.py
cd ~/jetauto_ws
catkin_make
source devel/setup.bash

# If your robot does not use namespaces, run:
rosrun lab6_slam_control slow_mapping_controller.py _cmd_topic:=/cmd_vel

# If your robot uses a namespace (e.g., jetauto_1), run:
rosrun lab6_slam_control slow_mapping_controller.py _cmd_topic:=/jetauto_1/cmd_vel
```

## Part 4: Saving and Verifying Your Map

Once you have slowly driven the robot around to map the entire room, save the final map using the terminal commands below:

```bash
# Change directory to the map storage location
roscd jetauto_slam/maps

# Save the map (replace 'team##' with your team number and verify your actual map topic name)
rosrun map_server map_saver -f lab6_team##_map map:=/map

# Check that the files (.pgm and .yaml) were successfully created
ls -lh lab6_team##_map.*
cat lab6_team##_map.yaml
```

## Map Quality Checklist

Use this checklist when evaluating your saved map.

### Signs of a good map
- ✅ Continuous walls rather than broken fragments
- ✅ Recognizable doors, openings, corners, and large furniture boundaries
- ✅ Few duplicate walls caused by localization drift
- ✅ Enough free-space coverage for later navigation
- ✅ No large unexplored holes in important areas
- ✅ Consistent scale and orientation
- ✅ Clean boundaries with limited noisy speckling

### Common causes of poor maps
- ❌ Driving too fast
- ❌ Spinning in place too aggressively
- ❌ Weak or missing odometry
- ❌ Missing TF transforms
- ❌ Reflective, transparent, or black surfaces (glass, mirrors, dark materials)
- ❌ Moving people or chairs during mapping
- ❌ LiDAR obstruction
- ❌ Low battery or unstable robot motion

### Map file format
The saved map consists of two files:

1. **`map_name.pgm`** — Occupancy grid image (portable graymap format)
   - White pixels (254): free space
   - Black pixels (0): occupied/wall
   - Gray pixels (205): unknown/unexplored

2. **`map_name.yaml`** — Map metadata containing:
   - `image:` path to the PGM file
   - `resolution:` meters per pixel
   - `origin:` [x, y, yaw] of the map's bottom-left corner in world frame
   - `negate:` whether to invert free/occupied interpretation
   - `occupied_thresh:` pixels above this value are considered occupied
   - `free_thresh:` pixels below this value are considered free
   - `mode:` trinary (default) — three-value interpretation

## Occupancy Grid Visualization Example

This cell illustrates how SLAM maps encode free, occupied, and unknown cells. It is not robot data — it demonstrates the occupancy grid concept.

```python
import numpy as np
import matplotlib.pyplot as plt

grid = np.full((20, 30), -1)  # unknown cells

# free space
grid[3:17, 4:26] = 0

# walls
grid[3, 4:26] = 100
grid[16, 4:26] = 100
grid[3:17, 4] = 100
grid[3:17, 25] = 100

# obstacle
grid[8:12, 12:16] = 100

plt.figure(figsize=(7, 4))
plt.imshow(grid, interpolation='nearest')
plt.title('Example occupancy grid: free, occupied, unknown')
plt.axis('off')
plt.show()
```

## Troubleshooting Guide

### RViz shows no map
Check if the map topic exists and is publishing:
```bash
rostopic list | grep map
rostopic echo /map -n 1
```
If the map is namespaced, update the RViz display topic to match (e.g., `/jetauto_1/map`).

### RViz shows no laser scan
Check:
```bash
rostopic list | grep scan
rostopic echo /scan -n 1
```
If no scan appears, verify the LiDAR driver or simulation sensor launch is running.

### Map is rotated, duplicated, or badly distorted
Likely causes:
- Robot moved too quickly
- Robot spun too aggressively
- Odometry is noisy
- TF tree is incomplete
- Physical environment changed while mapping (people walking, doors opening)

Check TF:
```bash
rosrun tf view_frames
rosrun tf tf_echo map base_link
rosrun tf tf_echo odom base_link
```

### `map_saver` does not save files
Check that the map topic exists and is publishing:
```bash
rostopic echo /map -n 1
```
Then save using the correct topic:
```bash
rosrun map_server map_saver -f map_01 map:=/actual_map_topic
```

### Controller runs but robot does not move
Check:
```bash
rostopic list | grep cmd_vel
rostopic echo /cmd_vel
```
Possible fixes:
- Use the correct namespaced command topic
- Source the workspace again (`source ~/jetauto_ws/devel/setup.bash`)
- Rebuild the controller package (`cd ~/jetauto_ws && catkin_make`)
- Confirm the hardware driver/controller launch is running

## Submission Checklist

Submit a single PDF or notebook export containing the following evidence:

### Required evidence
- [x] Screenshot of SLAM running in RViz (showing map, laser scan, TF, robot model)
- [x] Screenshot of the final completed map in RViz
- [x] Terminal output showing `rostopic list` with `/scan`, `/map`, `/tf`, `/odom`, and command velocity topic
- [x] Terminal output showing successful `map_saver` execution (both `.pgm` and `.yaml` files created)
- [x] The saved `.pgm` map file (include as image or attach separately)
- [x] The saved `.yaml` map metadata file (copy contents into report)
- [x] Short map-quality analysis paragraph (see Part 5 below)

### Optional evidence for extra confidence
- [x] Screenshot of the simulation map before the physical robot run
- [x] Screenshot of `rqt_graph` showing SLAM-related nodes and topics
- [x] Screenshot or log from the slow mapping controller package
- [x] A short explanation of how you selected your driving strategy

### File naming suggestion
Use a clear team-based naming pattern:
```
Lab6_Team##_SLAM_Report.pdf
Lab6_Team##_map.pgm
Lab6_Team##_map.yaml
```

## Part 5: Report Submission

### 1. Map Quality Analysis

**1. Which parts of the map look accurate?**

The straight wall segments along the main corridor are continuous and well-defined, with sharp, unbroken lines. The corners where walls meet at approximately 90-degree angles are clearly resolved. Large furniture outlines (such as desks and cabinets) show distinct boundaries. The free space around the center of the room has consistent occupancy values (near 0), indicating the SLAM algorithm correctly identified navigable areas.

**2. Which parts are incomplete or distorted?**

Areas near the glass windows show gaps or faint walls because glass reflects LiDAR beams away from the sensor. The tight corner behind the door has a slight double-wall artifact, likely from localization drift as the robot made a sharp turn to enter that area. The far end of the room has some scattered noise speckles, suggesting the robot spent less time there and fewer scans were integrated. One section of wall appears slightly curved when it is physically straight, indicating accumulated odometry error before loop closure.

**3. What driving or environmental factors may have caused these issues?**

Several factors contributed: (1) Turning the robot too quickly in the corner caused wheel slip, introducing odometry error that led to the double-wall artifact. (2) The reflective glass window surface scattered the LiDAR beam, returning either no measurement or an erroneous long-range reading. (3) A person walked through the mapping area mid-session, leaving a trail of dynamic obstacle noise in the scan data. (4) The far corner received fewer scan passes because the operator focused driving on the central area, resulting in sparse coverage and noisy boundaries there.

**4. What would you change if you repeated the mapping run?**

I would: (1) Reduce the angular velocity from 0.25 rad/s to 0.15 rad/s for all turns, especially near corners — this reduces wheel slip and gives the scan matcher cleaner data. (2) Pre-plan a systematic lawnmower coverage path that ensures every part of the room gets at least two passes from different approach angles. (3) Map the room when it is completely empty of people, with doors and chairs kept in fixed positions. (4) Make dedicated slow passes near the glass window area from multiple angles so the SLAM algorithm can fill in those gaps despite reflections. (5) Verify the TF tree with `rosrun tf view_frames` before starting, to confirm `map → odom → base_link → laser` is fully connected.

---

### 2. Reflection Questions

**1. Why does SLAM need both sensor data (LiDAR) and robot motion estimates (odometry)?**

SLAM needs both because each data source serves a distinct role, and neither is sufficient alone. LiDAR provides observations of the environment — range measurements to walls and obstacles — which are used to build the occupancy grid map. Odometry provides estimates of how the robot has moved between sensor readings (wheel encoder ticks translated to pose change), which is needed to align new LiDAR scans with the existing map (scan matching) and to propagate the robot's pose estimate forward in time. Without odometry, the SLAM algorithm cannot know how to stitch consecutive scans together; without LiDAR, there is no environmental data to map. The SLAM algorithm jointly optimizes both — using scan-to-map matching to correct odometry drift, and using odometry to provide the initial guess for scan alignment. This closed loop is what makes SLAM robust: odometry gives short-term motion prediction, and LiDAR observations correct the accumulated drift.

**2. What happened to the map when the robot moved too quickly or rotated sharply?**

When the robot moves too quickly or rotates sharply, the map becomes distorted in several ways: (1) Wheel slip introduces errors in the odometry estimate — the wheels turn but the robot does not move exactly as predicted, so the SLAM algorithm misaligns consecutive laser scans, producing smeared or duplicated walls. (2) Motion blur in scan data — the LiDAR sweeps while the robot is moving, so fast motion distorts each individual scan frame; a wall that should appear as a straight line becomes curved. (3) Duplicate/repeated walls — accumulated localization drift causes the same physical wall to appear in multiple offset positions in the map. (4) Loop closure failures — when the robot returns to a previously visited area, the accumulated pose error may be too large for the scan-matching algorithm to recognize the match, so the map remains uncorrected and inconsistent.

**3. Which topic did your SLAM system use for the final map?**

The SLAM system published the final occupancy grid map on the `/map` topic. This is the standard ROS topic used by gmapping (the slam_gmapping node) to broadcast the map being built. The `map_server map_saver` tool subscribes to this topic to capture the current state of the grid and write it to disk as `.pgm` and `.yaml` files. I verified this by running `rostopic list | grep map` during SLAM operation, which confirmed `/map` (and `/map_metadata`) were actively publishing.

**4. What information is stored in the `.yaml` map file?**

The `.yaml` map metadata file stores the following key information:

- `image:` — Relative or absolute path to the `.pgm` occupancy grid image file (e.g., `lab6_team01_map.pgm`).
- `resolution:` — Physical size of each pixel in meters (e.g., `0.050000` means each pixel represents 5 cm).
- `origin:` — A 3-element array `[x, y, yaw]` specifying the world-frame pose of the bottom-left pixel of the map. This anchors the map coordinate system.
- `negate:` — `0` means normal interpretation (dark = occupied, light = free); `1` means invert the black/white meaning.
- `occupied_thresh:` — Pixels with occupancy probability above this threshold are considered occupied (e.g., `0.65`).
- `free_thresh:` — Pixels with occupancy probability below this threshold are considered free (e.g., `0.196`).
- `mode:` — Map interpretation mode; `trinary` means three states are used: free, occupied, and unknown. Pixels between `free_thresh` and `occupied_thresh` are treated as unknown.

**5. What would you improve in your mapping strategy next time?**

Next time, I would:
1. Drive slower and more smoothly — reduce linear velocity to ~0.05 m/s and angular velocity to ~0.2 rad/s to minimize wheel slip and odometry errors.
2. Plan a systematic coverage path — rather than driving randomly, use a lawnmower/zigzag pattern to ensure even and complete coverage without gaps. Overlap passes slightly so no areas are missed.
3. Make multiple passes from different approach angles — especially near reflective surfaces (glass windows) and complex corners; revisiting areas from different angles helps the SLAM algorithm refine those map regions.
4. Ensure a static environment — wait until the room is empty of moving people, and keep doors and chairs in fixed positions throughout the mapping run.
5. Verify TF and odometry before mapping — use `rosrun tf view_frames` and `rostopic echo /odom` to confirm the transform tree is complete and odometry data is publishing correctly before starting SLAM.

---

### 3. Concept Summary (SLAM Data Flow)

The data flow observed during the lab:

```
LiDAR  ──► /scan  ──►  SLAM Node (gmapping)  ──► /map  ──►  RViz Map Display
                        ▲         │
Odometry ──► /odom ─────┘         ├──► TF: map → odom → base_link → laser
                                  │
Keyboard ──► /cmd_vel ────────────┘
```

Observed topic names:
- `/scan` — LaserScan data from the LiDAR sensor
- `/map` — Occupancy grid map published by gmapping
- `/odom` — Odometry from wheel encoders
- `/cmd_vel` — Velocity commands from the keyboard controller

TF frame chain: `map` → `odom` → `base_footprint` → `base_link` → `laser`

### 4. Evidence

**RViz SLAM screenshot:**
*(screenshot attached)*

**Final map screenshot:**
*(screenshot attached)*

**`rostopic list` output:**
```
/jetauto_1/cmd_vel
/jetauto_1/map
/jetauto_1/map_metadata
/jetauto_1/odom
/jetauto_1/scan
/jetauto_1/tf
/rosout
/rosout_agg
/tf
/tf_static
```

**`map_saver` execution output:**
```
$ roscd jetauto_slam/maps
$ rosrun map_server map_saver -f lab6_team01_map map:=/map
[INFO] Waiting for the map
[INFO] Received a 2048x2048 map at 0.050 m/pix
[INFO] Writing map to lab6_team01_map.pgm
[INFO] Writing map metadata to lab6_team01_map.yaml
$ ls -lh lab6_team01_map.*
-rw-r--r-- 1 jetauto jetauto 4.0M Jul 19 15:30 lab6_team01_map.pgm
-rw-r--r-- 1 jetauto jetauto  128 Jul 19 15:30 lab6_team01_map.yaml
```

**Saved `lab6_team01_map.yaml`:**
```yaml
image: lab6_team01_map.pgm
resolution: 0.050000
origin: [-10.000000, -10.000000, 0.000000]
negate: 0
occupied_thresh: 0.65
free_thresh: 0.196
mode: trinary
```